# Compare Python and MATLAB model outputs

This notebook compares the Python-generated `.mat` files in `output/` against the MATLAB reference `.mat` files in `outputs/`.

What it does:
- loads the `m` struct from each `.mat`
- compares all shared numeric fields
- separately prints the key physiological outputs used during validation

Note: `modelinputs.mat` is not used for strict equality here because MATLAB function handles and Python callables are serialized differently.

In [1]:
from pathlib import Path

import numpy as np
from scipy.io import loadmat


def find_repo_dir(start=None):
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src").exists() and (candidate / "outputs").exists():
            return candidate
    raise RuntimeError("Could not locate repo root")


REPO_DIR = find_repo_dir()
PYTHON_OUTPUT_DIR = REPO_DIR / "output"
MATLAB_OUTPUT_DIR = REPO_DIR / "outputs"

print(f"REPO_DIR={REPO_DIR}")
print(f"PYTHON_OUTPUT_DIR={PYTHON_OUTPUT_DIR}")
print(f"MATLAB_OUTPUT_DIR={MATLAB_OUTPUT_DIR}")

REPO_DIR=/Users/huaizefeng/Documents/GitHub/johnson-field-berry-2021-oeco
PYTHON_OUTPUT_DIR=/Users/huaizefeng/Documents/GitHub/johnson-field-berry-2021-oeco/output
MATLAB_OUTPUT_DIR=/Users/huaizefeng/Documents/GitHub/johnson-field-berry-2021-oeco/outputs


In [2]:
FOCUS_FIELDS = ("An_a", "JP700_a", "PAM4_a", "An_sa", "C_sa")


def load_mat_struct(mat_path, key):
    payload = loadmat(mat_path, squeeze_me=True, struct_as_record=False)
    return payload[key]


def struct_field_names(struct):
    if hasattr(struct, "_fieldnames"):
        return list(struct._fieldnames)
    return list(vars(struct).keys())


def max_abs_diff(reference_value, candidate_value):
    reference = np.asarray(reference_value, dtype=float)
    candidate = np.asarray(candidate_value, dtype=float)

    if reference.shape != candidate.shape:
        raise ValueError(f"shape mismatch: {reference.shape} != {candidate.shape}")

    if reference.size == 0:
        return 0.0

    both_nan = np.isnan(reference) & np.isnan(candidate)
    reference = np.where(both_nan, 0.0, reference)
    candidate = np.where(both_nan, 0.0, candidate)

    diff = np.abs(reference - candidate)
    if np.all(np.isnan(diff)):
        return 0.0

    return float(np.nanmax(diff))


def compare_structs(reference_struct, candidate_struct):
    numeric_diffs = []
    skipped_fields = []
    shape_mismatches = []

    shared_fields = sorted(
        set(struct_field_names(reference_struct)) & set(struct_field_names(candidate_struct))
    )

    for name in shared_fields:
        try:
            reference_value = getattr(reference_struct, name)
            candidate_value = getattr(candidate_struct, name)
            reference_array = np.asarray(reference_value, dtype=float)
            candidate_array = np.asarray(candidate_value, dtype=float)
        except (TypeError, ValueError):
            skipped_fields.append(name)
            continue

        if reference_array.shape != candidate_array.shape:
            shape_mismatches.append((name, reference_array.shape, candidate_array.shape))
            continue

        numeric_diffs.append((name, max_abs_diff(reference_array, candidate_array)))

    numeric_diffs.sort(key=lambda item: item[1], reverse=True)
    return numeric_diffs, skipped_fields, shape_mismatches


def compare_case(base_output_name, candidate_suffix="-py", key="m", top_n=10):
    reference_path = MATLAB_OUTPUT_DIR / base_output_name / f"{base_output_name}-modeloutputs.mat"
    candidate_path = (
        PYTHON_OUTPUT_DIR
        / f"{base_output_name}{candidate_suffix}"
        / f"{base_output_name}-modeloutputs.mat"
    )

    if not reference_path.exists():
        raise FileNotFoundError(reference_path)
    if not candidate_path.exists():
        raise FileNotFoundError(candidate_path)

    reference_struct = load_mat_struct(reference_path, key)
    candidate_struct = load_mat_struct(candidate_path, key)
    numeric_diffs, skipped_fields, shape_mismatches = compare_structs(
        reference_struct,
        candidate_struct,
    )

    print(f"[case] {base_output_name}{candidate_suffix}")
    print("[focus]")
    for name in FOCUS_FIELDS:
        if hasattr(reference_struct, name) and hasattr(candidate_struct, name):
            diff = max_abs_diff(getattr(reference_struct, name), getattr(candidate_struct, name))
            print(f"  {name}: {diff:.6g}")

    print("[top numeric diffs]")
    for name, diff in numeric_diffs[:top_n]:
        print(f"  {name}: {diff:.6g}")

    if skipped_fields:
        print(f"[skipped non-numeric] {len(skipped_fields)} fields")
    if shape_mismatches:
        print(f"[shape mismatches] {len(shape_mismatches)} fields")
        for name, reference_shape, candidate_shape in shape_mismatches[:top_n]:
            print(f"  {name}: {reference_shape} != {candidate_shape}")

    print()
    return {
        "base_output_name": base_output_name,
        "candidate_suffix": candidate_suffix,
        "numeric_diffs": numeric_diffs,
        "skipped_fields": skipped_fields,
        "shape_mismatches": shape_mismatches,
    }

In [3]:
DEFAULT_CASES = (
    ("Example1-Light-Type-I-C3-C4", "-py"),
    ("Example1-Light-NADP-ME-C4", "-py"),
    ("Example2-CO2-C3", "-py"),
    ("Example2-CO2-Type-I-C3-C4", "-py"),
    ("Example2-CO2-NADP-ME-C4", "-py"),
    ("Example3-Temperature-C3", "-py"),
    ("Example3-Temperature-Type-I-C3-C4", "-py"),
    ("Example3-Temperature-NADP-ME-C4", "-py"),
)

results = []
for base_output_name, candidate_suffix in DEFAULT_CASES:
    candidate_dir = PYTHON_OUTPUT_DIR / f"{base_output_name}{candidate_suffix}"
    if candidate_dir.exists():
        results.append(compare_case(base_output_name, candidate_suffix=candidate_suffix))

[case] Example1-Light-Type-I-C3-C4-py
[focus]
  An_a: 1.69407e-21
  JP700_a: 1.35525e-20
  PAM4_a: 6.66134e-15
  An_sa: 2.11758e-21
  C_sa: 7.58942e-19
[top numeric diffs]
  Kn2_sa: 0.000808239
  Kn2_ma: 1.23978e-05
  phi2N_sa: 1.51087e-13
  phi2P_sa: 1.33116e-13
  phi2n_sa: 1.09988e-13
  phi2p_sa: 6.92779e-14
  PAM3_a: 6.83897e-14
  phi2u_sa: 3.13083e-14
  phi2D_sa: 1.64868e-14
  phi2d_sa: 8.60423e-15
[skipped non-numeric] 1 fields

[case] Example1-Light-NADP-ME-C4-py
[focus]
  An_a: 1.17229e-18
  JP700_a: 1.35525e-19
  PAM4_a: 3.61933e-14
  An_sa: 1.17229e-18
  C_sa: 3.90963e-16
[top numeric diffs]
  Kn2_sa: 0.00100613
  Kn2_ma: 2.86102e-06
  PAM3_a: 6.03961e-13
  phi2N_sa: 1.56875e-13
  phi2n_sa: 1.30174e-13
  phi2P_sa: 1.00531e-13
  phi2u_sa: 7.55507e-14
  PAM5_a: 5.86753e-14
  phi2D_sa: 5.14588e-14
  PAM4_a: 3.61933e-14
[skipped non-numeric] 1 fields

[case] Example2-CO2-C3-py
[focus]
  An_a: 6.77626e-21
  JP700_a: 5.42101e-20
  PAM4_a: 2.22045e-16
  An_sa: 0
  C_sa: 0
[top numeri

In [4]:
# Custom one-off comparison
BASE_OUTPUT_NAME = "Example1-Light-Type-I-C3-C4"
CANDIDATE_SUFFIX = "-relayout"

compare_case(BASE_OUTPUT_NAME, candidate_suffix=CANDIDATE_SUFFIX)

[case] Example1-Light-Type-I-C3-C4-relayout
[focus]
  An_a: 1.69407e-21
  JP700_a: 1.35525e-20
  PAM4_a: 6.66134e-15
  An_sa: 2.11758e-21
  C_sa: 7.58942e-19
[top numeric diffs]
  Kn2_sa: 0.000808239
  Kn2_ma: 1.23978e-05
  phi2N_sa: 1.51087e-13
  phi2P_sa: 1.33116e-13
  phi2n_sa: 1.09988e-13
  phi2p_sa: 6.92779e-14
  PAM3_a: 6.83897e-14
  phi2u_sa: 3.13083e-14
  phi2D_sa: 1.64868e-14
  phi2d_sa: 8.60423e-15
[skipped non-numeric] 1 fields



{'base_output_name': 'Example1-Light-Type-I-C3-C4',
 'candidate_suffix': '-relayout',
 'numeric_diffs': [('Kn2_sa', 0.0008082389831542969),
  ('Kn2_ma', 1.239776611328125e-05),
  ('phi2N_sa', 1.51087475863676e-13),
  ('phi2P_sa', 1.3311574065255627e-13),
  ('phi2n_sa', 1.0998840727083348e-13),
  ('phi2p_sa', 6.927791673660977e-14),
  ('PAM3_a', 6.838973831690964e-14),
  ('phi2u_sa', 3.1308289294429414e-14),
  ('phi2D_sa', 1.6486811915683575e-14),
  ('phi2d_sa', 8.604228440844963e-15),
  ('PAM5_a', 7.646661082105766e-15),
  ('PAM4_a', 6.661338147750939e-15),
  ('Fmp_a', 1.6514567491299204e-15),
  ('Fmp_sa', 1.6503725469574348e-15),
  ('phi2F_sa', 1.4988010832439613e-15),
  ('PAM6_a', 9.159339953157541e-16),
  ('PAM1_a', 8.881784197001252e-16),
  ('phi2f_sa', 7.8236028766554e-16),
  ('PAM2_a', 6.661338147750939e-16),
  ('phi2n_ma', 5.551115123125783e-16),
  ('phi2N_ma', 4.440892098500626e-16),
  ('phi2u_ma', 3.3306690738754696e-16),
  ('phi1N_ma', 2.220446049250313e-16),
  ('phi1N_sa', 2